# R1 — CNN Double Descent

**Experiment**: Reproduce the model-wise double descent curve on a 5-layer CNN  
**Dataset**: CIFAR-10, 5 000-sample subset, 15% symmetric label noise  
**Sweep**: width multiplier k ∈ {1,2,4,6,8,12,16,24,32,48,64}, 2 seeds → 22 runs  
**Output**: `results/R1/fig1_r1_dd.png` (also saved to Drive if `USE_DRIVE=True`)

> **Estimated time**: ~8 min/run on Colab T4 → ~3 h total (runs sequentially).  
> Use multiple Colab accounts in parallel by splitting the `widths` list.


## 1  Environment setup

In [ ]:
# Install / verify dependencies
!pip install -q torch torchvision numpy matplotlib seaborn tqdm pandas
import torch
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import os

REPO_URL  = 'https://github.com/YOUR_USERNAME/project-6699.git'  # ← replace
REPO_DIR  = '/content/project-6699'
USE_DRIVE = True   # Set False to save locally inside Colab (lost on disconnect)

# ── Mount Drive ─────────────────────────────────────────────────────────────
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULT_DIR = '/content/drive/MyDrive/benign_overfitting/R1'
else:
    RESULT_DIR = f'{REPO_DIR}/results/R1'

os.makedirs(RESULT_DIR, exist_ok=True)
print(f'Results → {RESULT_DIR}')

# ── Clone / update repo ──────────────────────────────────────────────────────
if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

import sys
sys.path.insert(0, REPO_DIR)

## 2  (Optional) Run only a subset of widths in parallel

If you are splitting R1 across multiple Colab accounts, uncomment one block below before running the next cell.

In [ ]:
from run_r1 import R1_CONFIG, run_r1, plot_r1
import copy

cfg = copy.deepcopy(R1_CONFIG)

# ── Full sweep (default) ─────────────────────────────────────────────────────
# cfg['widths'] = [1, 2, 4, 6, 8, 12, 16, 24, 32, 48, 64]   # all 11 points

# ── Account A: narrow networks ───────────────────────────────────────────────
# cfg['widths'] = [1, 2, 4, 6, 8]    # first 5 points

# ── Account B: wide networks ─────────────────────────────────────────────────
# cfg['widths'] = [12, 16, 24, 32, 48, 64]   # last 6 points

print(f"Will train {len(cfg['widths']) * len(cfg['seeds'])} runs: k={cfg['widths']}")

## 3  Run R1

In [ ]:
# keep_alive: ping every 60 s so the Colab session doesn't idle out
import threading, time
def _keep_alive():
    while True:
        time.sleep(60)
        try:
            from google.colab.output import eval_js
            eval_js('0')
        except Exception:
            pass
_t = threading.Thread(target=_keep_alive, daemon=True)
_t.start()

results = run_r1(cfg, RESULT_DIR, resume=True)

## 4  Plot Figure 1

In [ ]:
plot_r1(RESULT_DIR)

# Inline display
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Image
fig_path = Path(RESULT_DIR) / 'fig1_r1_dd.png'
if fig_path.exists():
    display(Image(str(fig_path)))

## 5  Quick summary table

In [ ]:
import pandas as pd
from src.io_utils import load_results

results = load_results(RESULT_DIR, pattern='r1_*.json')
df = pd.DataFrame([{
    'k':           r['width_multiplier'],
    'seed':        r['seed'],
    'n_params':    r['n_params'],
    'train_err':   f"{r['train_error']:.3f}",
    'test_err':    f"{r['test_error']:.3f}",
    'wall_time_m': f"{r['wall_time_s']/60:.1f}",
} for r in results])
df = df.sort_values(['k', 'seed'])
display(df.to_string(index=False))